# Part 0 — Check your environment

Run this notebook **once at the start**, and again whenever something strange happens.
It checks that the SPINE container, the GPU, spine-prod and the example files are all
where the tutorial expects them, and it ends with a ✅/❌ summary.

**How to run a cell:** click it, then press **Shift-Enter**. While it runs you'll see `[*]`
next to the cell; when it finishes, a number appears.

> **Terminal equivalent** (from a compute node, see [`README.md`](README.md#path-b--the-terminal)):
> ```bash
> nvidia-smi
> spine_container.sh spine --version
> spine_container.sh spine --info
> git -C $SPINE_PROD_BASEDIR log --oneline -1
> ls -lh $TUTORIAL_DATA
> ```

## 1. Load the tutorial helpers

`tutorial_env.py` (in the tutorial's top folder) defines the same variables as
`setup_env.sh`: `WORKDIR`, `TUTORIAL_DATA`, `SPINE_CONFIG_PATH`, and so on. Every notebook starts with these lines.

In [ ]:
import sys; sys.path.insert(0, "..")
import tutorial_env as te
env = te.setup()
results = {}   # we'll collect check results here

**What you should see:** the host name (on S3DF something like `sdfampere007`),
`inside container: True`, and paths for `WORKDIR` and `TUTORIAL_DATA`.

If `inside container` is `False` on S3DF, the Jupyter session was not started with the
SPINE container. Recheck the "Commands to initiate Jupyter" box in Part 0, Path A.

## 2. Python packages

The container ships everything. Here we import each piece and print its version.
`MinkowskiEngine` (sparse convolutions), `torch` and `ROOT`/`larcv` exist only inside the container.

In [ ]:
import importlib

def version_of(module_name):
    try:
        module = importlib.import_module(module_name)
        return getattr(module, "__version__", "installed")
    except Exception as err:          # not installed, or broken
        return f"MISSING ({type(err).__name__})"

for name in ["spine", "numpy", "h5py", "pandas", "plotly", "torch", "MinkowskiEngine", "ROOT", "larcv"]:
    v = version_of(name)
    print(f"{name:<16} {v}")
    results[f"import {name}"] = not v.startswith("MISSING")

**What you should see:** `spine 1.4.0` and versions for everything else. On a laptop
(off S3DF), `torch`, `MinkowskiEngine`, `ROOT` and `larcv` are usually missing. That's fine
for Part 2b, which only reads output files.

## 3. GPU

SPINE's neural networks need a GPU to run at a sensible speed. `torch.cuda.is_available()`
asks PyTorch whether it can see one.

In [ ]:
results["GPU visible"] = te.has_gpu()
if results["GPU visible"]:
    import torch
    print("GPU:", torch.cuda.get_device_name(0))
    te.sh("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv", check=False)
else:
    print("No GPU visible. Fine for Part 2 (analysis); Parts 1 and 3 need one.")
    print("Jupyter: restart the session on partition 'ampere' with 1 GPU.")
    print("Terminal: run compute_shell.sh (it asks for a GPU on 'ampere').")

## 4. The `spine` command

`spine` is the program that runs SPINE from the command line. `--info` lists which optional parts work.

In [ ]:
results["spine command"] = te.which("spine") is not None
te.sh("spine --version && spine --info", check=False)

## 5. spine-prod (production configs)

On S3DF the tutorial uses the **standard spine-prod installation**,
`/sdf/data/neutrino/software/spine-prod`. That is the one used for **all official productions**.
A personal clone is only for development (Part 0, step 5.2).

In [ ]:
import os, subprocess
prod = os.environ["SPINE_PROD_BASEDIR"]
is_official = os.path.realpath(prod) == os.path.realpath(te.OFFICIAL_SPINE_PROD)
results["spine-prod found"] = os.path.isdir(os.path.join(prod, "config"))
if te.ON_S3DF:
    results["using the standard S3DF spine-prod"] = is_official

if results["spine-prod found"]:
    commit = subprocess.run(["git", "-C", prod, "log", "--oneline", "-1"],
                            capture_output=True, text=True).stdout.strip()
    version_file = os.path.join(prod, "DEFAULT_SPINE_VERSION")
    version = open(version_file).read().strip() if os.path.exists(version_file) else "?"
    print("spine-prod :", prod, "(standard S3DF install)" if is_official else "(personal copy)")
    print("commit     :", commit or "?")
    print("SPINE container version it expects:", version)
    if not commit.startswith("cc682a6"):
        print("note: this tutorial was written against cc682a6; a newer version should still work.")
else:
    print("spine-prod not found at", prod, "-> see Part 0 README step 5.2")
print("SPINE_CONFIG_PATH =", os.environ.get("SPINE_CONFIG_PATH"))
print("SPINE_CACHE_DIR   =", os.environ.get("SPINE_CACHE_DIR"))

## 6. Example files

`te.fetch()` downloads a file the first time, then reuses it. If you followed step 5.7 of
the README, nothing is downloaded here.

In [ ]:
for name, kind in [("generic_small.root", "larcv"),
                   ("protodune-sp_small.root", "larcv"),
                   ("generic_small_spine.h5", "reco")]:
    try:
        te.fetch(name, kind)
        results[f"file {name}"] = True
    except Exception as err:
        print("could not get", name, ":", err)
        results[f"file {name}"] = False

## 7. Account and pre-downloaded weights

In [ ]:
account = os.environ.get("SPINE_ACCOUNT", "")
print("SPINE_ACCOUNT =", account or "<not set: edit 00_s3df_basics/scripts/setup_env.sh>")
results["SPINE_ACCOUNT set"] = bool(account)

cache = os.environ["SPINE_CACHE_DIR"]
ckpts = [f for f in os.listdir(cache) if f.endswith(".ckpt")] if os.path.isdir(cache) else []
print(f"{len(ckpts)} weight file(s) cached in {cache}")
results["weights pre-downloaded"] = len(ckpts) >= 2

## 8. Summary

In [ ]:
for check, ok in results.items():
    print(("✅" if ok else "❌"), check)
print()
print("Off S3DF, ❌ for torch/MinkowskiEngine/ROOT/larcv/GPU/weights is expected.")

**What you should see on S3DF in a GPU session:** all ✅. In a CPU (milano) session,
only `GPU visible` should be ❌.

Next: **[Part 1 — processing files through SPINE](../01_processing/README.md)**.